In [ ]:
# Import Required Libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Dataset

url = "https://raw.githubusercontent.com/ditikrushna/Predict-Sales-Revenue-Using-Multiple-Regression-Model/master/Advertising.csv"

df = pd.read_csv(url)

if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Question 1(a)
# Compute Correlation Matrix and Identify Strongest Correlation with Sales

corr_matrix = df.corr()

print("Correlation Matrix")
print(corr_matrix)

print("\nCorrelation with Sales:")
print(corr_matrix['sales'].sort_values(ascending=False))

strongest = corr_matrix['sales'].drop('sales').idxmax()

print("\nAnswer:")
print(f"{strongest} has the strongest correlation with Sales.")

In [ ]:
# Question 1(b)
# Scatter Plots of Sales vs Predictors

fig, axes = plt.subplots(1,3, figsize=(15,5))

sns.scatterplot(data=df,x='TV',y='sales',ax=axes[0])
axes[0].set_title('TV vs Sales')

sns.scatterplot(data=df,x='radio',y='sales',ax=axes[1])
axes[1].set_title('Radio vs Sales')

sns.scatterplot(data=df,x='newspaper',y='sales',ax=axes[2])
axes[2].set_title('Newspaper vs Sales')

plt.show()

print("Answer:")
print("TV and Radio show strong positive linear relationships with Sales.")
print("Newspaper shows a weak relationship.")

In [ ]:
# Question 1(c)
# Check Outliers

print(df.describe())

sns.boxplot(data=df)
plt.show()

print("Answer:")
print("No major outliers are observed that would severely affect the model.")

In [ ]:
# Question 2
# Fit Multiple Linear Regression Model

model = smf.ols('sales ~ TV + radio + newspaper', data=df).fit()

print(model.summary())

print("\nRegression Equation:")
print(f"Sales = {model.params['Intercept']:.4f} + {model.params['TV']:.4f}*TV + {model.params['radio']:.4f}*Radio + {model.params['newspaper']:.4f}*Newspaper")

print("\nR-Squared =", model.rsquared)

print("\nInterpretation:")
print("TV coefficient indicates expected increase in Sales for every $1000 increase in TV budget.")
print("Radio coefficient indicates expected increase in Sales for every $1000 increase in Radio budget.")
print("Newspaper coefficient indicates expected increase in Sales for every $1000 increase in Newspaper budget.")

In [ ]:
# Question 3
# Model Significance and Individual Tests

print("Overall F-Test")
print("F Statistic =", model.fvalue)
print("P Value =", model.f_pvalue)

print("\nIndividual Predictor Tests")

for var in ['TV','radio','newspaper']:
    print("\n",var)
    print("t-stat =", model.tvalues[var])
    print("p-value =", model.pvalues[var])

print("\nAnswer:")
print("TV and Radio are significant.")
print("Newspaper is not statistically significant at α = 0.05.")

In [ ]:
# Question 4(a)
# Train-Test Split and Error Metrics

X = df[['TV','radio','newspaper']]
y = df['sales']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

train_model = smf.ols('sales ~ TV + radio + newspaper', data=pd.concat([X_train,y_train],axis=1)).fit()

pred = train_model.predict(X_test)

mse = mean_squared_error(y_test,pred)
rmse = np.sqrt(mse)

print("MSE =", mse)
print("RMSE =", rmse)

In [ ]:
# Question 4(b)
# Residual Diagnostics

residuals = train_model.resid
fitted = train_model.fittedvalues

plt.figure(figsize=(8,5))
sns.scatterplot(x=fitted,y=residuals)
plt.axhline(0,color='red')
plt.title("Residuals vs Fitted")
plt.show()

sm.qqplot(residuals,line='45')
plt.show()

print("Answer:")
print("Residuals appear randomly distributed.")
print("Normality assumption is reasonably satisfied.")
print("No major heteroscedasticity observed.")

In [ ]:
# Question 5
# Predictions and Recommendation

new1 = pd.DataFrame({'TV':[200],'radio':[40],'newspaper':[20]})
new2 = pd.DataFrame({'TV':[100],'radio':[20],'newspaper':[50]})

pred1 = model.predict(new1)[0]
pred2 = model.predict(new2)[0]

print("Prediction A =", round(pred1,2))
print("Prediction B =", round(pred2,2))

print("\nRecommendation:")
print("Invest more in TV and Radio advertising.")
print("These variables have significant coefficients and strong correlation with Sales.")
print("Newspaper advertising contributes very little after controlling for TV and Radio.")